# Act 1: Destination Selection via Exoplanet Transit Photometry

Exoplanets are detected when they orbit in front of their host star, causing periodic dips in observed brightness called transits. We use the Box Least Squares (BLS) algorithm to scan for candidate orbital periods and durations across the light curve. A Random Forest classifier then examines folded transit shape features to distinguish genuine exoplanets from stellar noise. Tuning tree depth and detection thresholds balances false alarms against missed planetary targets.

In [ ]:
# Setup and package installation
REPO_URL = "https://github.com/placeholder/meridian"
# Option 1: Install from GitHub in Google Colab
# !pip install -q git+$REPO_URL

# Option 2: Install from local editable package
# !pip install -e /path/to/meridian-workshop

import sys
from pathlib import Path
repo_dir = Path.cwd()
while repo_dir.parent != repo_dir and not (repo_dir / "meridian").exists():
    repo_dir = repo_dir.parent
if (repo_dir / "meridian").exists() and str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))


In [ ]:
import os
TEAM_ID = os.environ.get("MERIDIAN_TEAM_ID", "your-team-name")

from meridian.state import load_state
from meridian.ui import briefing, dashboard, debrief
state = load_state(TEAM_ID)


In [ ]:
briefing(1)

### Building the Stellar Survey

We load training light curves and a catalog of candidate stars for our probe to scout. Each curve records brightness over a 27-day observation window. The survey extracts 46 transit features including BLS power spectra and phase-folded flux profiles.

In [ ]:
from meridian.act1_exoplanet import (
    build_survey, train, rank_candidates, choose_target,
    plot_candidates, plot_lightcurve, human_vs_ai, reveal
)
survey = build_survey(TEAM_ID)
print(f'Survey ready: {len(survey.candidate_star_ids)} candidates to evaluate.')

### Human vs AI Warm-up

Can human eyes detect faint transit dips faster than an automated algorithm? Inspect the candidate curves below, make your prediction, and compare against baseline machine learning predictions. Notice how stellar noise and flares obscure true planetary signals.

In [ ]:
# Inspect 4 candidate light curves and benchmark against AI
sample_ids = human_vs_ai(survey, n=4)
reveal(guesses=[1, 0, 1, 0], survey=survey, star_ids=sample_ids)

### Inspecting Feature Extraction

The Box Least Squares (BLS) algorithm fits a box-shaped transit model across thousands of trial periods. The peak in the periodogram reveals the most probable orbital period. Folding the time-series at this period stacks multiple transits together, boosting signal-to-noise.

In [ ]:
import matplotlib.pyplot as plt
from meridian.act1_exoplanet.generator import TIME_GRID
plot_lightcurve(TIME_GRID, survey.candidate_fluxes[0], title=f'Candidate Star {survey.candidate_star_ids[0]}')
plt.show()

### YOUR TURN: Tune the Transit Classifier

Adjust your Random Forest hyperparameters and detection probability threshold below. A lower threshold flags more candidate planets but risks selecting a false positive star. Your highest-ranked candidate will become Meridian's official destination!

In [ ]:
# YOUR TURN: Adjust Act 1 Random Forest hyperparameters
knobs = {
    "n_estimators": 300,
    "max_depth": None,
    "feature_set": "full",
    "threshold": 0.5,
}

In [ ]:
# Train classifier and select primary exoplanet target
model = train(survey, **knobs)
ranked = rank_candidates(model, survey)
target_star_id = ranked.iloc[0]['star_id']
state = choose_target(state, survey, model, target_star_id)
print(f'Target Selected: {target_star_id} | Score: {state.act1.score:.1f}/200 | AUC: {state.act1.auc:.3f}')

In [ ]:
# Display top candidate rankings
plot_candidates(ranked, top_n=10)
plt.show()

In [ ]:
debrief(state, 1)

In [ ]:
state.save()
dashboard(state)